# Onset rendering — KS method × excitation (forward spectrograms)

Single pluck, onset stepping 0.1 s/frame (0→3.9 s, fs=16000, 4 s). Paper params
`{f0, a1, g, p, gain, dyn} = {110 Hz, 0.2, 0.99, 0.25, 0.9, 0.9}`, onset `t` swept.

2×2 grid — each KS method with STE vs its **own** matched fractional excitation:

|              | STE (integer fwd) | own fractional excitation |
|--------------|-------------------|----------------------------|
| **tKSA** (time-domain) | tKSA · STE | tKSA · tEXC (Lagrange, time) |
| **fKSA** (freq-sampling, LTI 8 s) | fKSA · STE | fKSA · fEXC (phase, freq)  |

Cyan line = onset. Watch for: the faint wrapped tail in the fKSA row, and any
excitation pre-echo (energy just *before* the onset) in the fractional columns.

In [ ]:
import sys
from pathlib import Path
_root = Path.cwd()
while not (_root / "src").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import Image, display

from src.synths.synth import Synth, SynthConfig
from src.synths.ddsp import Implementation, ExcitationMode

FS = 16000; DUR = 4.0; N = int(FS * DUR); F0 = 110.0; G = 0.99
PAR = dict(a1=0.2, pluck_position=0.25, burst_gain=0.9, dynamic_level=0.9)

def mk(impl, exc, **kw):
    return Synth(SynthConfig(num_samples=N, fs=FS, implementation=impl,
                             excitation_mode=exc, **kw))
TD, FSAMP = Implementation.TIME_DOMAIN, Implementation.FREQUENCY_SAMPLING
STE, tEXC, fEXC = (ExcitationMode.STE, ExcitationMode.TIME_DOMAIN_LAGRANGE,
                   ExcitationMode.FREQUENCY_SAMPLING)
# 2x2: rows = KS method, cols = {STE, own fractional excitation}
PANELS = [["tKSA · STE", mk(TD, STE)],
          ["tKSA · tEXC (time)", mk(TD, tEXC)],
          ["fKSA · STE", mk(FSAMP, STE, use_lti=True, lti_pad_factor=2)],
          ["fKSA · fEXC (freq)", mk(FSAMP, fEXC, use_lti=True, lti_pad_factor=2)]]

def render(synth, onset_s):
    t = float(onset_s) / DUR
    p = {"exists": torch.ones(1, 1), "time": torch.tensor([[t]], dtype=torch.float32), "f0": torch.full((1, 1), F0),
         "decay": torch.full((1, 1), G), **{k: torch.full((1, 1), v) for k, v in PAR.items()}}
    with torch.no_grad():
        y, _ = synth(p)
    return y[0].numpy()

ONSETS = np.arange(0.0, DUR, 0.1)
NFFT, HOP = 1024, 256; win = torch.hann_window(NFFT)
def spec_db(y):
    S = torch.stft(torch.from_numpy(y).float(), n_fft=NFFT, hop_length=HOP,
                   window=win, return_complex=True).abs()
    return 20.0 * np.log10(S.numpy() + 1e-6)
print("configs:", [p[0] for p in PANELS], "| frames:", len(ONSETS))


In [ ]:
specs = {name: [spec_db(render(syn, o)) for o in ONSETS] for name, syn in PANELS}

vmin, vmax, fmax = -60.0, 20.0, 3000.0
extent = [0, DUR, 0, FS / 2]
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
axf = axes.ravel(); ims = []
for ax, (name, _) in zip(axf, PANELS):
    im = ax.imshow(specs[name][0], origin="lower", aspect="auto", extent=extent,
                   vmin=vmin, vmax=vmax, cmap="magma")
    ax.set_ylim(0, fmax); ax.set_title(name); ims.append(im)
for ax in axes[-1]: ax.set_xlabel("time (s)")
for ax in axes[:, 0]: ax.set_ylabel("frequency (Hz)")
lines = [ax.axvline(0.0, color="cyan", ls="--", lw=1.2) for ax in axf]
sup = fig.suptitle(""); fig.tight_layout()
names = [p[0] for p in PANELS]
def update(k):
    for im, name in zip(ims, names): im.set_data(specs[name][k])
    for ln in lines: ln.set_xdata([ONSETS[k], ONSETS[k]])
    sup.set_text(f"onset = {ONSETS[k]:.1f} s   (g={G}, f0={F0:.0f} Hz)")
    return ims + lines + [sup]
anim = animation.FuncAnimation(fig, update, frames=len(ONSETS), interval=150, blit=False)
GIF = "onset_ks_excitation_grid.gif"
anim.save(GIF, writer=animation.PillowWriter(fps=8)); plt.close(fig)
print("saved", GIF)
display(Image(filename=GIF))
